#### Setup

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.paths import load_ccep, runs, electrodes_tsv

cc = load_ccep()
print(len(cc), "patients")

74 patients


###  All electrodes of all patients in one table

In [2]:
E = pd.concat(
    [electrodes_tsv(p["id"], p["ses"]).assign(patient=p["id"]) for p in cc],
    ignore_index=True,
)
print(E.shape)
for col in ["soz", "resected", "edge", "group"]:
    print(col, E[col].value_counts(dropna=False).to_dict())

(8655, 16)
soz {'no': 8302, 'yes': 353}
resected {'no': 7999, 'yes': 656}
edge {'no': 8482, 'yes': 173}
group {'grid': 4410, 'other': 3123, 'strip': 920, 'depth': 202}


#### Per patient: how many SOZ, resected and edge electrodes

In [3]:
yes = lambda s: (s == "yes").sum()

per = E.groupby("patient").agg(
    n_electrodes=("name", "size"),
    n_soz=("soz", yes),
    n_resected=("resected", yes),
    n_edge=("edge", yes),
)
per["soz_annotated"] = per["n_soz"] > 0
per["has_resection"] = per["n_resected"] > 0
per

,n_electrodes,n_soz,n_resected,n_edge,soz_annotated,has_resection
patient,,,,,,
sub-ccepAgeUMCU01,133,0,0,0,False,False
sub-ccepAgeUMCU02,96,0,0,0,False,False
sub-ccepAgeUMCU03,133,0,0,0,False,False
sub-ccepAgeUMCU04,128,0,0,0,False,False
sub-ccepAgeUMCU05,133,0,0,0,False,False
...,...,...,...,...,...,...
sub-ccepAgeUMCU70,68,0,0,0,False,False
sub-ccepAgeUMCU71,68,0,0,0,False,False
sub-ccepAgeUMCU72,133,0,0,0,False,False


Patients: SOZ marked? Resection marked?

In [4]:
pd.crosstab(per["soz_annotated"], per["has_resection"], margins=True)

has_resection,False,True,All
soz_annotated,,,
False,33,5,38
True,1,35,36
All,34,40,74


Electrodes: SOZ vs. resected

In [5]:
pd.crosstab(E["soz"], E["resected"], margins=True)

resected,no,yes,All
soz,,,
no,7887,415,8302
yes,112,241,353
All,7999,656,8655


SOZ electrodes that were NOT removed (left in the brain)

In [6]:
E["soz_not_resected"] = (E["soz"] == "yes") & (E["resected"] == "no")
per["n_soz_not_resected"] = E.groupby("patient")["soz_not_resected"].sum()

per[per["n_soz_not_resected"] > 0].sort_values("n_soz_not_resected", ascending=False)

,n_electrodes,n_soz,n_resected,n_edge,soz_annotated,has_resection,n_soz_not_resected
patient,,,,,,,
sub-ccepAgeUMCU58,133,37,23,4,True,True,22
sub-ccepAgeUMCU65,133,18,8,6,True,True,13
sub-ccepAgeUMCU47,133,11,0,0,True,False,11
sub-ccepAgeUMCU61,133,19,20,16,True,True,9
sub-ccepAgeUMCU07,130,24,48,8,True,True,8
sub-ccepAgeUMCU48,93,9,2,10,True,True,7
sub-ccepAgeUMCU42,133,20,51,6,True,True,7
sub-ccepAgeUMCU36,133,7,13,8,True,True,6
sub-ccepAgeUMCU62,133,11,6,2,True,True,5


How many N1 responses and stimulated pairs touch flagged electrodes?

In [7]:
rows = []
for p in cc:
    el = electrodes_tsv(p["id"], p["ses"]).set_index("name")
    for r in runs(p):
        rec = el.reindex(r["channel_names"])   # match by name, not by position
        n1 = r["n1_peak_sample"]
        for flag in ["soz", "resected"]:
            flagged = rec[flag].eq("yes").values
            rows.append(dict(
                patient=p["id"],
                run=r["runName"],
                flag=flag,
                rec_channels_flagged=flagged.sum(),
                n1_at_flagged=int(np.sum(~np.isnan(n1[flagged, :]))),
                n1_total=int(np.sum(~np.isnan(n1))),
                stim_pairs_flagged=sum(
                    el.reindex(pair.split("-"))[flag].eq("yes").any()
                    for pair in r["stimpair_names"]
                ),
                stim_pairs_total=len(r["stimpair_names"]),
            ))

R = pd.DataFrame(rows)
summary = R.groupby("flag")[["rec_channels_flagged", "n1_at_flagged", "n1_total",
                             "stim_pairs_flagged", "stim_pairs_total"]].sum()
summary["share_n1_lost_%"] = (100 * summary["n1_at_flagged"] / summary["n1_total"]).round(1)
summary

,rec_channels_flagged,n1_at_flagged,n1_total,stim_pairs_flagged,stim_pairs_total,share_n1_lost_%
flag,,,,,,
resected,1024,4134,40168,690,4907,10.3
soz,528,2404,40168,415,4907,6.0


In [8]:
rows = []
for p in cc:
    el = electrodes_tsv(p["id"], p["ses"])
    n1s = [r["n1_peak_sample"] for r in runs(p)]
    rows.append(dict(
        patient=p["id"],
        age=p["age"],
        soz_annotated=(el["soz"] == "yes").any(),
        has_resection=(el["resected"] == "yes").any(),
        n_electrodes=len(el),
        n_depth=(el["group"] == "depth").sum(),
        n_stim_pairs=sum(a.shape[1] for a in n1s),
        n1_rate=np.mean(np.concatenate([(~np.isnan(a)).ravel() for a in n1s])),
    ))

P = pd.DataFrame(rows)
print(P.groupby("soz_annotated").size())
P.groupby("soz_annotated")[["age", "n_electrodes", "n_depth", "n_stim_pairs", "n1_rate"]].median().round(3)

soz_annotated
False    38
True     36
dtype: int64


,age,n_electrodes,n_depth,n_stim_pairs,n1_rate
soz_annotated,,,,,
False,16.5,133.0,0.0,63.0,0.073
True,17.5,133.0,0.0,55.5,0.067
